In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Pump–probe EOM integrator sweeping the FIRST interval tau1.
Saves complex arrays for post-processing.

Outputs:
  alpha11n1_pp.npy, alpha1n11_pp.npy, alphan111_pp.npy, alpha001_pp.npy
  times_pp.npy, tau1_values.npy, meta_pp.json
"""

import json
import numpy as np
from scipy.integrate import solve_ivp

In [2]:
# -------------------
# Parameters 
# -------------------
kappa = 1.1
gamma_phi1= 0.6
gamma_phi2= 0.9
gamma = 0.0
Numb = 100.0
g12 = 0.185
g23 = np.sqrt(2)*g12*(1-0.22)  # pump–probe example: turn off 2–3 coupling

tau2 = 12.0            # fixed second pulse arrival time
tau3 = 30.0           # fixed third pulse arrival time
tauw = 0.2             # pulse width

tmax = 500.0
tplot = 0.2            # sampling for saved output

omega_c = 198.3
omega_l = 198.3
Delta_c = omega_l - omega_c

omega2 = 198.3
omega3 = 198.3+196.8
Delta_2 = omega_l - omega2
Delta_3 = 2.0*omega_l - omega3

# Sweep of the FIRST interval τ1
tau1_start = 2.0
tau1_end   = tau2
tau1_values = np.arange(tau1_start, tau1_end + 1e-12, 0.05)

# Time grid
times = np.arange(0.0, tmax + 1e-12, tplot)

# -------------------
# Pulse envelope
# -------------------
def f(x):
    return np.exp(-(x**2)/tauw)

In [3]:
# -------------------
# Variable ordering (24 variables total) – same as before
# -------------------
#  0  α100,   1  α010,   2  α001,   3  α11n1,   4  α1n11,   5  αn111,
#  6  ρ100,   7  ρ010,   8  ρ001,   9  ρ1n10,  10  ρn110,  11  ρ10n1,
# 12  ρn101,  13  ρ01n1, 14  ρ0n11, 15  ρ110,  16  ρ101,  17  ρ011,
# 18  ρ21ph11n1, 19  ρ21ph1n11, 20  ρ21phn111,
# 21  ρ32ph11n1, 22  ρ32ph1n11, 23  ρ32phn111

A100, A010, A001, A11n1, A1n11, An111 = range(6)
R100, R010, R001, R1n10, Rn110, R10n1, Rn101, R01n1, R0n11, R110, R101, R011, R21_11n1, R21_1n11, R21_n111, R32_11n1, R32_1n11, R32_n111 = \
    6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23

# -------------------
# RHS with τ1 as the looped parameter (τ2, τ3 fixed)
# -------------------
def rhs(t, y, tau1):
    i = 1j
    α100, α010, α001, α11n1, α1n11, αn111 = y[A100], y[A010], y[A001], y[A11n1], y[A1n11], y[An111]
    ρ100, ρ010, ρ001 = y[R100], y[R010], y[R001]
    ρ1n10, ρn110, ρ10n1 = y[R1n10], y[Rn110], y[R10n1]
    ρn101, ρ01n1, ρ0n11 = y[Rn101], y[R01n1], y[R0n11]
    ρ110, ρ101, ρ011 = y[R110], y[R101], y[R011]
    ρ21ph11n1, ρ21ph1n11, ρ21phn111 = y[R21_11n1], y[R21_1n11], y[R21_n111]
    ρ32ph11n1, ρ32ph1n11, ρ32phn111 = y[R32_11n1], y[R32_1n11], y[R32_n111]

    α100c, α010c, α001c = np.conj(α100), np.conj(α010), np.conj(α001)

    dy = np.zeros_like(y, dtype=np.complex128)

    # α-fields (drive on first/second/third pulses at t ~ τ1, τ2, τ3)
    dy[A100]  = -((kappa/2) - i*Delta_c)*α100 - i*Numb*g12*ρ100 - f(t - tau1)
    dy[A010]  = -((kappa/2) - i*Delta_c)*α010 - i*Numb*g12*ρ010 - f(t - tau2)
    dy[A001]  = -((kappa/2) - i*Delta_c)*α001 - i*Numb*g12*ρ001 - f(t - tau3)

    dy[A11n1] = -((kappa/2) - i*Delta_c)*α11n1 - i*Numb*g12*ρ21ph11n1 - i*Numb*g23*ρ32ph11n1
    dy[A1n11] = -((kappa/2) - i*Delta_c)*α1n11 - i*Numb*g12*ρ21ph1n11 - i*Numb*g23*ρ32ph1n11
    dy[An111] = -((kappa/2) - i*Delta_c)*αn111 - i*Numb*g12*ρ21phn111 - i*Numb*g23*ρ32phn111

    # single-quantum coherences
    dy[R100] = -((gamma_phi1/2) - i*Delta_2)*ρ100 - i*g12*α100
    dy[R010] = -((gamma_phi1/2) - i*Delta_2)*ρ010 - i*g12*α010
    dy[R001] = -((gamma_phi1/2) - i*Delta_2)*ρ001 - i*g12*α001

    # population-like cross terms
    dy[R1n10] = -gamma*ρ1n10 - i*g12*α100*np.conj(ρ010) + i*g12*α010c*ρ100
    dy[Rn110] = -gamma*ρn110 - i*g12*α010*np.conj(ρ100) + i*g12*α100c*ρ010
    dy[R10n1] = -gamma*ρ10n1 - i*g12*α100*np.conj(ρ001) + i*g12*α001c*ρ100
    dy[Rn101] = -gamma*ρn101 - i*g12*α001*np.conj(ρ100) + i*g12*α100c*ρ001
    dy[R01n1] = -gamma*ρ01n1 - i*g12*α010*np.conj(ρ001) + i*g12*α001c*ρ010
    dy[R0n11] = -gamma*ρ0n11 - i*g12*α001*np.conj(ρ010) + i*g12*α010c*ρ001

    # double-quantum coherences
    dy[R110] = -(gamma_phi1+gamma_phi2 - i*Delta_3)*ρ110 - i*g23*α100*ρ010 - i*g23*α010*ρ100
    dy[R101] = -(gamma_phi1+gamma_phi2  - i*Delta_3)*ρ101 - i*g23*α100*ρ001 - i*g23*α001*ρ100
    dy[R011] = -(gamma_phi1+gamma_phi2  - i*Delta_3)*ρ011 - i*g23*α010*ρ001 - i*g23*α001*ρ010

    # ρ21ph*
    dy[R21_11n1] = -(0.5*gamma_phi1 - i*Delta_2)*ρ21ph11n1 - i*g12*α11n1 \
                   + 2*i*g12*α100*ρ01n1 + 2*i*g12*α010*ρ10n1 - i*g23*α001c*ρ110
    dy[R21_1n11] = -(0.5*gamma_phi1 - i*Delta_2)*ρ21ph1n11 - i*g12*α1n11 \
                   + 2*i*g12*α100*ρ0n11 + 2*i*g12*α001*ρ1n10 - i*g23*α010c*ρ101
    dy[R21_n111] = -(0.5*gamma_phi1 - i*Delta_2)*ρ21phn111 - i*g12*αn111 \
                   + 2*i*g12*α010*ρn101 + 2*i*g12*α001*ρn110 - i*g23*α100c*ρ011

    # ρ32ph* (kept for completeness; g23=0 here)
    dy[R32_11n1] = -(0.5*gamma_phi2 - i*(Delta_3 - Delta_2))*ρ32ph11n1 \
                   - i*g23*α010*ρ10n1 - i*g23*α100*ρ01n1 + i*g12*α001c*ρ110
    dy[R32_1n11] = -(0.5*gamma_phi2 - i*(Delta_3 - Delta_2))*ρ32ph1n11 \
                   - i*g23*α001*ρ1n10 - i*g23*α100*ρ0n11 + i*g12*α010c*ρ101
    dy[R32_n111] = -(0.5*gamma_phi2 - i*(Delta_3 - Delta_2))*ρ32phn111 \
                   - i*g23*α001*ρn110 - i*g23*α010*ρn101 + i*g12*α100c*ρ011

    return dy


In [4]:
# -------------------
# Integrate for each tau1 with robust sampling & guards
# -------------------
n_t = len(times)
n_tau1 = len(tau1_values)

alpha11n1 = np.zeros((n_t, n_tau1), dtype=np.complex128)
alpha1n11 = np.zeros((n_t, n_tau1), dtype=np.complex128)
alphan111 = np.zeros((n_t, n_tau1), dtype=np.complex128)
alpha001  = np.zeros((n_t, n_tau1), dtype=np.complex128)

y0 = np.zeros(24, dtype=np.complex128)

print(f"Integrating {n_tau1} trajectories (tau1 from {tau1_values[0]} to {tau1_values[-1]}) ...")
for j, tau1 in enumerate(tau1_values):
    sol = solve_ivp(
        fun=lambda t, y: rhs(t, y, tau1),
        t_span=(0.0, tmax),
        y0=y0,
        method="BDF",
        rtol=1e-7,
        atol=1e-9,
        t_eval=times,           # exact grid sampling (prevents interpolant gaps)
        max_step=tplot          # keep steps aligned-ish with output cadence
    )
    if not sol.success:
        print(f"[warn] solver failed for tau1={tau1}: {sol.message}")

    # sol.y has shape (24, n_t) aligned with `times`
    Yt = sol.y
    alpha11n1[:, j] = Yt[A11n1, :]
    alpha1n11[:, j] = Yt[A1n11, :]
    alphan111[:, j] = Yt[An111, :]
    alpha001[:,  j] = Yt[A001,  :]

    # Scrub non-finite entries in this τ1 column 
    for arr in (alpha11n1, alpha1n11, alphan111, alpha001):
        col = arr[:, j]
        bad = ~np.isfinite(col)
        if np.any(bad):
            col[bad] = 0.0

# -------------------
# Save results (pump–probe suffix _pp)
# -------------------
np.save("alpha11n1_pp.npy", alpha11n1)
np.save("alpha1n11_pp.npy", alpha1n11)
np.save("alphan111_pp.npy", alphan111)
np.save("alpha001_pp.npy",  alpha001)

np.save("times_pp.npy", times.astype(np.float64))
np.save("tau1_values.npy", tau1_values.astype(np.float64))

meta = {
    "mode": "pump-probe",
    "loop_param": "tau1",
    "kappa": kappa,
    "gamma_phi1": gamma_phi1,
    "gamma_phi2": gamma_phi2,
    "gamma": gamma,
    "Numb": Numb,
    "g12": g12,
    "g23": float(g23),
    "tau1_start": float(tau1_start),
    "tau1_end": float(tau1_end),
    "tau2": float(tau2),
    "tau3": float(tau3),
    "tauw": float(tauw),
    "tmax": float(tmax),
    "tplot": float(tplot),
    "omega_c": float(omega_c),
    "omega_l": float(omega_l),
    "Delta_c": float(Delta_c),
    "omega2": float(omega2),
    "omega3": float(omega3),
    "Delta_2": float(Delta_2),
    "Delta_3": float(Delta_3)
}
with open("meta_pp.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved: *_pp.npy, times_pp.npy, tau1_values.npy, meta_pp.json")

Integrating 201 trajectories (tau1 from 2.0 to 11.999999999999964) ...
Saved: *_pp.npy, times_pp.npy, tau1_values.npy, meta_pp.json


In [5]:
# Check that 24 variables...first check 1d fourier transform